# 02 — Baseline

Load the sampled books, clean and prepare the text, split into train/test, and score a Zero-R baseline.

## Setup

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt

from src import config

BOOKS_DIR = config.RAW_DIR / "books"

## Load the sampled books

In [2]:
sample = pd.read_csv(config.INTERIM_DIR / "sample.csv")
sample["text"] = [
    (BOOKS_DIR / f"{book_id}.txt").read_text(encoding="utf-8", errors="replace")
    for book_id in sample["id"]
]
sample["n_words"] = sample["text"].str.split().str.len()
sample[["title", "period", "n_words"]].head()

,title,period,n_words
0,Halleck's New English Literature by Reuben P. ...,pre-1750,167833
1,The Alchemist by Ben Jonson,pre-1750,48264
2,A Short View of the Immorality and Profaneness...,pre-1750,62215
3,Poems of Henry Vaughan Silurist Volume II by H...,pre-1750,71940
4,The Submarine Boys for the Flag by Victor G. D...,pre-1750,46602


Some of these books are labelled incorrectly on the timeframe they were published, e.g. *Halleck's New English Literature*
(a 1900s textbook) and *The Submarine Boys for the Flag* (1910s) are both labelled pre-1750.
This is investigated below in **Check for wrong publication dates**.

## Clean footnotes and editorial notes

In [3]:
import re

def clean_text(text):
    text = re.sub(r"\[[^\]]*\]", " ", text)                    # remove anything in [square brackets]
    text = re.sub(r"FOOTNOTES?:", " ", text, flags=re.IGNORECASE)  # remove "FOOTNOTE:" / "FOOTNOTES:" headings
    return text

sample["text"] = sample["text"].apply(clean_text)
sample["n_words"] = sample["text"].str.split().str.len()

Cleaning the sources of footnotes and editorial notes: anything in [square brackets] and "FOOTNOTES" headings are removed.
Why this is needed is shown in **Check for remaining footnotes** below.

## Inspect the texts

In [4]:
print(sample["text"].iloc[1][:1500])





Produced by Amy E Zelmer, Robert Prince, Sue Asscher





THE ALCHEMIST


By Ben Jonson




INTRODUCTION

The greatest of English dramatists except Shakespeare, the first
literary dictator and poet-laureate, a writer of verse, prose,
satire, and criticism who most potently of all the men of his time
affected the subsequent course of English letters: such was Ben
Jonson, and as such his strong personality assumes an interest to
us almost unparalleled, at least in his age.

Ben Jonson came of the stock that was centuries after to give to
the world Thomas Carlyle; for Jonson's grandfather was of
Annandale, over the Solway, whence he migrated to England.
Jonson's father lost his estate under Queen Mary, "having been cast
into prison and forfeited."  He entered the church, but died a
month before his illustrious son was born, leaving his widow and
child in poverty.  Jonson's birthplace was Westminster, and the
time of his birth early in 1573.  He was thus nearly ten years
Shakespeare's 

In [5]:
print(sample["n_words"].describe())
sample.groupby("period")["n_words"].median()

count       500.000000
mean      70706.678000
std       77394.954804
min         442.000000
25%       20318.000000
50%       50598.500000
75%       94135.500000
max      818001.000000
Name: n_words, dtype: float64


period
1750-1799    39565.5
1800-1849    56352.5
1850-1899    65337.0
1900-1918    55329.5
pre-1750     29846.5
Name: n_words, dtype: float64

No Gutenberg licence, but some books have introductions written after the original publication,
e.g. *The Alchemist* (1610) starts with an introduction written around 1900.

Book lengths vary a lot (442 to 818,001 words), and by period (median about 30,000 words for pre-1750 vs about 65,000 for 1850–1899).
The model could learn "short book = old" instead of learning the language. That's the shortcut we want to avoid, since it does not answer our research question.
Possible fix is taking a chunk out of the middle of every book. Would have to take into account possible instances of notes added after publication.
Also solves the issues of introductions/forewords written after publication.

## Remove short books

In [6]:
(sample["n_words"] < 5000).sum()

np.int64(23)

In [7]:
sample.loc[sample["n_words"] < 5000, "period"].value_counts()

period
1800-1849    9
1750-1799    8
1850-1899    3
pre-1750     2
1900-1918    1
Name: count, dtype: int64

Checking which books have less than 5000 words, which is the amount we want to sample.
23 books are shorter, mostly from 1750–1849, so keeping them would link short excerpts with those periods.

In [8]:
sample = sample[sample["n_words"] >= 5000].reset_index(drop=True)
sample["period"].value_counts().sort_index()

period
1750-1799    92
1800-1849    91
1850-1899    97
1900-1918    99
pre-1750     98
Name: count, dtype: int64

Removing works that are shorter than 5000 words, so excerpt length can't reveal the period. 477 books remain (91–99 per period).

## Take a 5,000-word excerpt from the middle of each book

In [10]:
def middle_chunk(text, n_words=5000):
    words = text.split()
    if len(words) <= n_words:
        return " ".join(words)
    start = (len(words) - n_words) // 2
    return " ".join(words[start:start + n_words])

sample["excerpt"] = sample["text"].apply(middle_chunk)
sample["excerpt"].str.split().str.len().describe()

count     477.0
mean     5000.0
std         0.0
min      5000.0
25%      5000.0
50%      5000.0
75%      5000.0
max      5000.0
Name: excerpt, dtype: float64

Creates a small reusable tool. It splits a book into words, finds the middle, and keeps 5,000 words from there.
Every excerpt is now exactly 5,000 words (std 0).

In [11]:
print(sample["excerpt"].iloc[1][:1000])

Not always necessary: The children of perdition are oft-times Made instruments even of the greatest works: Beside, we should give somewhat to man's nature, The place he lives in, still about the fire, And fume of metals, that intoxicate The brain of man, and make him prone to passion. Where have you greater atheists than your cooks? Or more profane, or choleric, than your glass-men? More antichristian than your bell-founders? What makes the devil so devilish, I would ask you, Sathan, our common enemy, but his being Perpetually about the fire, and boiling Brimstone and arsenic? We must give, I say, Unto the motives, and the stirrers up Of humours in the blood. It may be so, When as the work is done, the stone is made, This heat of his may turn into a zeal, And stand up for the beauteous discipline, Against the menstruous cloth and rag of Rome. We must await his calling, and the coming Of the good spirit. You did fault, t' upbraid him With the brethren's blessing of Heidelberg, weighing 

Checked an excerpt by eye: The Alchemist excerpt is original 1610 play text, not the modern introduction.

## Check for wrong publication dates

A book can't mention a year after it was written. For each excerpt, find the latest year mentioned and compare it with the label.
Books whose excerpt mentions a year more than 20 years after their label are flagged (20 allows for small slips, e.g. a book
published a few years after it was written).

In [12]:
def latest_year(text):
    years = [int(y) for y in re.findall(r"\b(1[4-9]\d\d)\b", text) if int(y) <= 1950]
    return max(years) if years else 0

sample["latest_year"] = sample["excerpt"].apply(latest_year)
sample["year_gap"] = sample["latest_year"] - sample["publication_date"]

flagged = sample[sample["year_gap"] > 20].sort_values("year_gap", ascending=False)
print(len(flagged), "books flagged")
print(flagged["period"].value_counts().sort_index())
flagged[["title", "publication_date", "latest_year", "period"]]

31 books flagged
period
1750-1799     7
1800-1849     2
1850-1899     3
pre-1750     19
Name: count, dtype: int64


,title,publication_date,latest_year,period
17,Rembrandt with a Complete List of His Etchings...,1637,1912,pre-1750
28,Initial Studies in American Letters by Henry A...,1610,1884,pre-1750
89,Chronological Retrospect of the History of Yar...,1618,1883,pre-1750
33,St. John's College Cambridge by Robert Forsyth...,1616,1869,pre-1750
56,The Bow Its History Manufacture and Use by Hen...,1676,1921,pre-1750
70,Notes and Queries Vol. V Number 134 May 22 185...,1619,1851,pre-1750
22,History of the Scottish expedition to Norway i...,1612,1837,pre-1750
19,The Man of Taste by James Bramston,1731,1950,pre-1750
81,Two Poems Against Pope by Welsted Guerinot and...,1735,1950,pre-1750
97,The Harlot's Progress by Theophilus Cibber and,1733,1948,pre-1750


In [13]:
sample = sample[sample["year_gap"] <= 20].reset_index(drop=True)
sample["period"].value_counts().sort_index()

period
1750-1799    85
1800-1849    89
1850-1899    94
1900-1918    99
pre-1750     79
Name: count, dtype: int64

31 books flagged, 19 of them pre-1750. Many PG-19 dates are the date of the book's subject, not when it was written
(e.g. *Rembrandt with a Complete List of His Etchings*, labelled 1637, mentions 1912). Some are genuinely old texts in modern reprints
whose excerpt includes modern editorial text (e.g. *The Man of Taste*, 1731, mentions 1950).
All flagged books are removed, leaving 446 (79–99 per period).
Limitation: mislabelled books that mention no years are not caught, so some label noise remains, especially in pre-1750.

## Check for remaining footnotes

In [14]:
markers = ["[Footnote", "Footnote", "Transcriber", "Editor's note", "[Note"]
for m in markers:
    n = sample["excerpt"].str.contains(m, case=False, regex=False).sum()
    print(f"{m!r}: {n} excerpts")

'[Footnote': 0 excerpts
'Footnote': 6 excerpts
'Transcriber': 1 excerpts
"Editor's note": 0 excerpts
'[Note': 0 excerpts


Notes within the text e.g. footnotes, editor's notes. Cleaned up at the start: before cleaning, about 10% of excerpts mentioned
footnotes; after cleaning, 6 of 446 (about 1.3%).
Limitations: footnote text written outside brackets is not removed, and some original text in brackets (e.g. stage directions) is lost.

In [15]:
ex = sample[sample["excerpt"].str.contains("Footnote", case=False)].iloc[0]
i = ex["excerpt"].lower().find("footnote")
print(ex["title"], "|", ex["period"])
print(ex["excerpt"][i - 200 : i + 400])

A Vindication of Rights of Woman by Mary Wollstonecraft | 1750-1799
seriously on this curious passage, and I could produce many similar ones; and some, so very sentimental, that I have heard rational men use the word indecent, when they mentioned them with disgust. (*Footnote. Can you?--Can you? would be the most emphatical comment, were it drawled out in a whining voice.) Throughout there is a display of cold, artificial feelings, and that parade of sensibility which boys and girls should be taught to despise as the sure mark of a little vain mind. Florid appeals are made to heaven, and to the BEAUTEOUS INNOCENTS, the fairest images of heaven here below, whil


The remaining example is Wollstonecraft's own footnote, written by the author rather than a later editor, so it is fine to keep.
Older books may also have Latin passages (e.g. Henry Vaughan's poems). These are left in as a limitation.

## Train/test split grouped by author

In [16]:
author = sample["title"].str.extract(r".*\sby\s(.+)$", flags=2)[0].str.strip()

unknown = author.isna() | author.str.lower().isin(["anonymous", "various", "unknown"])
sample["group"] = author.where(~unknown, sample["title"])

sample["group"].value_counts().head(10)

group
Ben Jonson             5
Honore de Balzac       4
William Shakespeare    3
Anthony Trollope       3
Molire                 2
David Hume             2
Joseph Trapp           2
Daniel Defoe           2
Henry Fielding         2
William Hogarth        2
Name: count, dtype: int64

Listing authors that have more than one text. Books with no known author (Anonymous, Various, Unknown) are each treated as their own group,
since they are not one person.

In [17]:
from sklearn.model_selection import StratifiedGroupKFold

splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=config.RANDOM_STATE)
train_idx, test_idx = next(splitter.split(sample, sample["period"], groups=sample["group"]))

train = sample.iloc[train_idx]
test = sample.iloc[test_idx]
print("Train:", len(train), " Test:", len(test))

Train: 357  Test: 89


Works are grouped by author so that they are always on the same side of train/test, and stratified so each period has the same share in both.
357 books in train, 89 in test.

In [18]:
print("Authors in both train and test:", len(set(train["group"]) & set(test["group"])))
pd.DataFrame({
    "train": train["period"].value_counts(normalize=True),
    "test": test["period"].value_counts(normalize=True),
}).sort_index().round(2)

Authors in both train and test: 0


,train,test
period,,
1750-1799,0.19,0.19
1800-1849,0.20,0.19
1850-1899,0.21,0.21
1900-1918,0.22,0.22
pre-1750,0.18,0.18


Confirming no author's works are in both train and test (0), and each period is a similar share of both sets.

## Zero-R baseline

In [19]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import cross_validate

baseline = make_pipeline(
    TfidfVectorizer(),
    DummyClassifier(strategy="most_frequent"),
)

cv = StratifiedGroupKFold(n_splits=config.CV_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
scores = cross_validate(
    baseline, train["excerpt"], train["period"],
    groups=train["group"], cv=cv, scoring=["accuracy", "f1_macro"],
)

print(f"Accuracy: {scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}")
print(f"Macro-F1: {scores['test_f1_macro'].mean():.3f} ± {scores['test_f1_macro'].std():.3f}")

Accuracy: 0.221 ± 0.005
Macro-F1: 0.072 ± 0.001


Scoring the baseline: Zero-R ignores the text and always predicts the most common period.
It is scored with 5-fold cross-validation on the training set only, grouped by author. The test set is not used until the end of the project.

The accuracy is 0.221 ± 0.005 because the model always predicts the most common period (1900–1918), which is about 22% of the books.
Macro-F1 is very low (0.072 ± 0.001) because the other 4 periods are never predicted.
The score barely changes across the 5 folds so the result is stable.
The bar for Milestone 3 is to beat the 0.221 accuracy and the 0.072 macro-F1 to show it has learned from the language.

## Summary and limitations

- 500 sampled books loaded; footnotes and editorial notes in [square brackets] removed.
- 23 books under 5,000 words removed, so that length can't reveal the period.
- Each book reduced to a 5,000-word excerpt from the middle, which skips modern introductions and end notes.
- 31 books whose excerpt mentions years far after their label removed, leaving 446 books (79–99 per period).
- Split 80/20 grouped by author and stratified by period: 357 train, 89 test, no author in both.
- Zero-R baseline (5-fold grouped CV on train): accuracy 0.221 ± 0.005, macro-F1 0.072 ± 0.001.

**Limitations:**
- Some wrong labels remain (books that mention no years are not checked).
- Footnotes outside brackets are not removed; some original bracketed text (e.g. stage directions) is lost.
- Some older books contain Latin passages, and some are translations.
- Author names are taken from titles, so spelling differences (e.g. "E. Phillips Oppenheim" vs "E. Philips Oppenheim")
  count as different authors, and accented letters can be lost (e.g. "Molire" for Molière).